# Anime Recommender · Notebook 1 — Data preparation

本 notebook 完成:
- 改用 **hernan4444/anime-recommendation-database-2020** 資料集 (有 synopsis、更多 metadata)
- 額外保留 `synopsis` 欄位給後續多模態文字嵌入 (Notebook 04) 使用
- 欄位名稱統一重新命名,讓 Notebook 2/3 完全不用改

執行流程一樣:Drive → Kaggle API → 下載資料 → EDA → 清理 → 切分 → 存 artifacts。

> 在 Colab 上點 **Runtime → Run all** 一次跑完。預估 5–10 分鐘。

## Step 1 — 掛載 Google Drive 並建立 artifacts 資料夾

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ARTIFACTS_DIR = '/content/drive/MyDrive/anime-recsys/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f'Artifacts will be saved to: {ARTIFACTS_DIR}')

## Step 2 — Kaggle API token (uploaded once, then reused)

In [ ]:
import os, shutil
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('請上傳你的 kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API ready ✅')

## Step 3 — 下載新資料集

hernan4444/anime-recommendation-database-2020 比 CooperUnion 那份大,但我們只需要其中幾個檔案:
- `anime_with_synopsis.csv` ── anime metadata + synopsis (我們的多模態文字來源)
- `anime.csv` ── 完整 metadata
- `rating_complete.csv` ── user-anime 評分

In [ ]:
!pip install -q kaggle
!mkdir -p /content/data
!cd /content/data && kaggle datasets download -d hernan4444/anime-recommendation-database-2020 --unzip --force
!ls -lh /content/data | head -20

## Step 4 — 載入資料 + 欄位重新命名(對齊舊 pipeline)

In [ ]:
import pandas as pd
import numpy as np

# === Synopsis 來源 ===
# anime_with_synopsis.csv 欄位: MAL_ID, Name, Score, Genres, sypnopsis (注意:資料集打錯字 sypnopsis 不是 synopsis)
syn = pd.read_csv('/content/data/anime_with_synopsis.csv')
print(f'anime_with_synopsis: {syn.shape}, columns: {list(syn.columns)}')
# 處理欄位名差異
syn_col = 'sypnopsis' if 'sypnopsis' in syn.columns else 'synopsis'

# === 完整 metadata ===
anime = pd.read_csv('/content/data/anime.csv')
print(f'anime: {anime.shape}, columns: {list(anime.columns)[:15]}...')

In [ ]:
# 統一欄位名 (對齊舊 pipeline 使用的:anime_id, name, genre, type, episodes, rating, members, synopsis)
anime = anime.rename(columns={
    'MAL_ID': 'anime_id',
    'Name': 'name',
    'Genres': 'genre',
    'Type': 'type',
    'Episodes': 'episodes',
    'Score': 'rating',
    'Members': 'members',
})

# 合併 synopsis
syn = syn.rename(columns={'MAL_ID': 'anime_id', syn_col: 'synopsis'})
anime = anime.merge(syn[['anime_id', 'synopsis']], on='anime_id', how='left')

# 只保留需要的欄位
keep_cols = ['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members', 'synopsis']
anime = anime[[c for c in keep_cols if c in anime.columns]].copy()

# 處理一些非預期值
anime['rating'] = pd.to_numeric(anime['rating'], errors='coerce')
anime['members'] = pd.to_numeric(anime['members'], errors='coerce').fillna(0).astype(int)
anime['synopsis'] = anime['synopsis'].fillna('').astype(str)

print(f'清理後 anime shape: {anime.shape}')
print(f'有 synopsis 的比例: {(anime["synopsis"].str.len() > 50).mean():.2%}')
anime.head(2)

In [ ]:
# === Ratings ===
# rating_complete.csv 欄位: user_id, anime_id, rating
rating = pd.read_csv('/content/data/rating_complete.csv')
print(f'rating: {rating.shape}, columns: {list(rating.columns)}')
rating.head()

## Step 5 — EDA

### 5.1 評分分佈

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
rating['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#1F4E79')
ax.set_title('Rating distribution (hernan4444 2020)')
ax.set_xlabel('rating'); ax.set_ylabel('count')
plt.tight_layout(); plt.show()

### 5.2 動漫類型分佈

In [ ]:
from collections import Counter
gc = Counter()
for g in anime['genre'].dropna():
    for t in g.split(','):
        if t.strip():
            gc[t.strip()] += 1
top_genres = pd.Series(gc).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5))
top_genres.plot(kind='bar', ax=ax, color='#2E75B6')
ax.set_title('Top 20 Genres')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()
print(f'總共 {len(gc)} 種 genre')

### 5.3 Synopsis 長度分佈 (確認我們有足夠文字做語意嵌入)

In [ ]:
synopsis_len = anime['synopsis'].str.split().str.len()
print(synopsis_len.describe())
synopsis_len.clip(upper=400).hist(bins=50, figsize=(8,3))
plt.title('Synopsis length (words, clipped at 400)')
plt.xlabel('# words'); plt.ylabel('# anime')
plt.tight_layout(); plt.show()

## Step 6 — Cleaning & filtering

- 移除無效評分
- 只保留評過 >= 5 部的 user
- 只保留被 >= 5 個 user 評過的 anime

In [ ]:
df = rating[(rating['rating'] > 0) & (rating['rating'] <= 10)].copy()

for _ in range(2):
    user_counts = df.groupby('user_id').size()
    df = df[df['user_id'].isin(user_counts[user_counts >= 5].index)]
    item_counts = df.groupby('anime_id').size()
    df = df[df['anime_id'].isin(item_counts[item_counts >= 5].index)]

print(f'過濾後:{len(df):,} 評分 / {df["user_id"].nunique():,} users / {df["anime_id"].nunique():,} animes')

In [ ]:
# 抽樣以加快訓練 (hernan4444 太大,訓練全量需要很久)
# 隨機抽 50000 個 user 即可,評估時樣本仍足夠且有代表性
MAX_USERS = 50000
rng = np.random.RandomState(42)
users = df['user_id'].unique()
if len(users) > MAX_USERS:
    sampled = rng.choice(users, size=MAX_USERS, replace=False)
    df = df[df['user_id'].isin(sampled)]
# 再次過濾低活動 item
item_counts = df.groupby('anime_id').size()
df = df[df['anime_id'].isin(item_counts[item_counts >= 5].index)]
print(f'抽樣後:{len(df):,} 評分 / {df["user_id"].nunique():,} users / {df["anime_id"].nunique():,} animes')

## Step 7 — 建立 id 對照表

In [ ]:
user_ids = sorted(df['user_id'].unique())
anime_ids = sorted(df['anime_id'].unique())
user_id_to_idx = {u: i for i, u in enumerate(user_ids)}
idx_to_user_id = {i: u for u, i in user_id_to_idx.items()}
anime_id_to_idx = {a: i for i, a in enumerate(anime_ids)}
idx_to_anime_id = {i: a for a, i in anime_id_to_idx.items()}
print(f'#users = {len(user_ids):,}, #anime = {len(anime_ids):,}')

## Step 8 — Per-user 80/20 切分

In [ ]:
from sklearn.model_selection import train_test_split
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_parts, test_parts = [], []
for uid, g in df.groupby('user_id', sort=False):
    if len(g) < 5:
        train_parts.append(g)
        continue
    tr, te = train_test_split(g, test_size=0.2, random_state=42)
    train_parts.append(tr); test_parts.append(te)
train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)
print(f'train: {len(train_df):,}  test: {len(test_df):,}')

## Step 9 — 建立 user_history (基於 train)

In [ ]:
user_history = train_df.groupby('user_id')['anime_id'].apply(set).to_dict()
print(f'user_history 涵蓋 {len(user_history):,} 位 user')

## Step 10 — 挑 10 位 Demo User

In [ ]:
rpu = train_df.groupby('user_id').size().sort_values(ascending=False)
cands = rpu[(rpu >= 30) & (rpu <= 200)]
demo_users = cands.head(10).index.tolist()
print('Demo users:', demo_users)

## Step 11 — 儲存所有 artifacts

重要:`anime_meta.parquet` 這次**多了 synopsis 欄位**,Notebook 04 會用它做嵌入。

In [ ]:
import pickle, json

# anime metadata (只保留我們資料集裡的 anime + 帶 synopsis)
anime_meta = anime[anime['anime_id'].isin(anime_ids)].copy()
anime_meta = anime_meta[['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members', 'synopsis']]
anime_meta.to_parquet(f'{ARTIFACTS_DIR}/anime_meta.parquet', index=False)

train_df.to_parquet(f'{ARTIFACTS_DIR}/train.parquet', index=False)
test_df.to_parquet(f'{ARTIFACTS_DIR}/test.parquet', index=False)

with open(f'{ARTIFACTS_DIR}/mappings.pkl', 'wb') as f:
    pickle.dump({
        'user_id_to_idx': user_id_to_idx,
        'idx_to_user_id': idx_to_user_id,
        'anime_id_to_idx': anime_id_to_idx,
        'idx_to_anime_id': idx_to_anime_id,
    }, f)

with open(f'{ARTIFACTS_DIR}/user_history.pkl', 'wb') as f:
    pickle.dump(user_history, f)

with open(f'{ARTIFACTS_DIR}/demo_users.json', 'w') as f:
    json.dump([int(u) for u in demo_users], f)

print('✅ Saved to:', ARTIFACTS_DIR)
!ls -lh {ARTIFACTS_DIR}

## ✅ 完成

下一步:
1. **接著跑 Notebook 02** (三個傳統模型,完全不用改)
2. **跑 Notebook 03** (評估,完全不用改)
3. **跑 Notebook 04** (新的多模態文字嵌入)